In [ ]:
# 프로세스(Process): 실행 중인 프로그램 하나를 의미합니다. 운영체제로부터 메모리 공간을 독립적으로 할당받는 작업 단위입니다.
# 스레드(Thread): 프로세스 안에서 실행되는 **"실행 흐름의 단위"**입니다. 하나의 프로세스는 최소 하나 이상의 스레드를 가지며, 
# 여러 개의 스레드는 프로세스 내의 자원(메모리 등)을 공유하며 동시에 작업을 수행할 수 있습니다.

# 병행성, Concurrency) : 레드를 나누어 사용하면 "동시에 여러 일이 일어나는 것처럼" 보이게 할 수 있습니다.

# 파이썬 스레드의 특징: GIL (Global Interpreter Lock)
# GIL은 한 번에 하나의 스레드만 파이썬 바이트코드를 실행하도록 제한하는 잠금 장치입니다.
# 파이썬의 스레드는 **CPU 연산이 많은 작업(CPU-bound)**에는 큰 효과가 없지만, 
# 입출력 작업(I/O-bound)(파일 읽기/쓰기, 네트워크 통신, DB 접근 등)에는 매우 효과적입니다.

In [ ]:
# 두 개의 작업(Task)이 동시에 돌아가는 상황을 가정해 보겠습니다. 하나는 2초가 걸리는 작업이고, 다른 하나는 3초가 걸리는 작업입니다.

import threading
import time

# 1. 실행할 작업(함수) 정의
def print_numbers(name, delay):
    print(f"[{name}] 작업 시작")
    count = 0
    while count < 5:
        print(f"[{name}] 실행 중... ({count})")
        time.sleep(delay)  # 지정된 시간 동안 대기 (I/O 작업 시뮬레이션)
        count += 1
    print(f"[{name}] 작업 완료!")

############################################################################
# --- 일반적인 순차 실행 (Single Thread) ---
print("=== 순차 실행 시작 ===")
start_time = time.time()
print_numbers("Task A", 1)
print_numbers("Task B", 1)
end_time = time.time()
print(f"총 소요 시간 (순차): {end_time - start_time:.2f}초\n")

############################################################################
# --- 스레드를 이용한 병렬 실행 (Multi-Thread) ---
print("=== 스레드 실행 시작 ===")
start_time = time.time()

# 2. 스레드 객체 생성
# target: 실행할 함수, args: 함수에 전달할 인자(튜플 형태)
thread1 = threading.Thread(target=print_numbers, args=("Thread-1", 1))
thread2 = threading.Thread(target=print_numbers, args=("Thread-2", 1))

# 3. 스레드 시작
# 스레드를 실제로 실행시키는 명령입니다. 호출하는 순간 새로운 실행 흐름이 생성됩니다.
thread1.start()
thread2.start()

# 4. 스레드가 종료될 때까지 기다림 (Join)
# join()을 호출하지 않으면, 메인 프로그램은 스레드가 끝날 때까지 기다리지 않고 바로 종료될 수 있습니다.
thread1.join()
thread2.join()

end_time = time.time()
print(f"총 소요 시간 (스레드): {end_time - start_time:.2f}초")


=== 순차 실행 시작 ===
[Task A] 작업 시작
[Task A] 실행 중... (0)
[Task A] 실행 중... (1)
[Task A] 실행 중... (2)
[Task A] 실행 중... (3)
[Task A] 실행 중... (4)
[Task A] 작업 완료!
[Task B] 작업 시작
[Task B] 실행 중... (0)
[Task B] 실행 중... (1)
[Task B] 실행 중... (2)
[Task B] 실행 중... (3)
[Task B] 실행 중... (4)
[Task B] 작업 완료!
총 소요 시간 (순차): 10.03초

=== 스레드 실행 시작 ===
[Thread-1] 작업 시작
[Thread-1] 실행 중... (0)
[Thread-2] 작업 시작
[Thread-2] 실행 중... (0)
[Thread-2] 실행 중... (1)[Thread-1] 실행 중... (1)

[Thread-1] 실행 중... (2)[Thread-2] 실행 중... (2)

[Thread-1] 실행 중... (3)[Thread-2] 실행 중... (3)

[Thread-2] 실행 중... (4)
[Thread-1] 실행 중... (4)
[Thread-2] 작업 완료!
[Thread-1] 작업 완료!
총 소요 시간 (스레드): 5.02초


In [ ]:
# 순차 실행: Task A가 5초(1초씩 5번) 걸리고, 그 뒤에 Task B가 5초 걸리므로 총 약 10초가 걸립니다.
# 스레드 실행: 두 작업이 동시에 돌아가므로, 가장 오래 걸리는 작업의 시간인 약 5초 만에 모든 작업이 완료됩니다.

# 파이썬에서의 구현: 파이썬에서 진짜 '병렬' 처리를 하고 싶다면 threading 대신 multiprocessing 모듈을 사용해야 합니다. 
# 프로세스를 나누면 CPU 코어마다 독립적인 메모리와 GIL을 갖게 되어 진짜로 동시에 일을 할 수 있습니다.

# 1. 병행성 (Concurrency): "동시에 하는 것처럼 보이게 함"
# CPU가 하나의 스레드를 아주 잠깐 실행하고, 바로 다른 스레드로 넘어가서 실행하는 과정을 반복합니다. 이를 **컨텍스트 스위칭(Context Switching)**이라고 합니다.
# 특징: 실제로는 한 번에 하나의 작업만 수행하고 있지만, 사람의 눈에는 작업들이 마치 동시에 일어나는 것처럼 보입니다.
# 파이썬의 스레드 (GIL 때문): 파이썬은 **GIL(Global Interpreter Lock)**이라는 제약 때문에, CPU 연산(계산) 작업에 
# 대해서는 한 번에 하나의 스레드만 실행할 수 있습니다. 즉, CPU가 매우 빠르게 스레드 사이를 왔다 갔다 하며 "동시에 하는 것처럼" 보이게 만듭니다.

# 2. 병렬성 (Parallelism): "진짜로 동시에 실행함"
# "여러 명이 각각 맡은 일을 동시에 수행하는 것"
# 방식: 여러 개의 CPU 코어(Core)가 각각 서로 다른 스레드를 맡아, 물리적으로 같은 순간에 서로 다른 명령어를 실행합니다.
# 특징: 작업이 실제로 물리적으로 분리되어 동시에 진행됩니다.
# 파이썬에서의 구현: 파이썬에서 진짜 '병렬' 처리를 하고 싶다면 threading 대신 multiprocessing 모듈을 사용해야 합니다. 
# 프로세스를 나누면 CPU 코어마다 독립적인 메모리와 GIL을 갖게 되어 진짜로 동시에 일을 할 수 있습니다.


# I/O 작업(파일 읽기, 네트워크 대기 등)을 할 때: 스레드를 쓰면 "어? 하나가 기다리는 동안 다른 게 실행되네?" 
# 동시에 실행되는 느낌(병행성)을 얻어 효율이 올라감.
# CPU 연산(복잡한 수학 계산 등)을 할 때: 파이썬 스레드는 "어? 하나가 계산하는 동안 다른 건 기다리네?" 

# 실제로 동시에 실행되지 못하고 번갈아 가며 수행됨(병행성만 존재).
# 따라서, **"어떤 작업을 하느냐에 따라 진짜 동시에 실행될 수도 있고, 동시에 하는 것처럼 보일 뿐일 수도 있다"**가 가장 정확한 답변입니다.